In [ ]:
%load_ext autoreload
%autoreload 2

import sys
import torch

sys.path.append('.')
from src.utils import *
from src.visualization import *
from src.engine import *
from src.dataset import *

In [ ]:
set_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
NUM_WORKERS = 0  # Windows/Jupyter에서는 0이 가장 안정적
print(f"Device: {device} | num_workers: {NUM_WORKERS}")

run_dirs = make_run_dir('./outputs')

In [ ]:
DATA_DIR = './data/new_k-fold_data/train'
CLASS_NAMES = ['good', 'type1', 'type2', 'type3', 'type4', 'type5']

all_paths, all_labels_np = get_image_paths_and_labels(DATA_DIR, CLASS_NAMES)

In [ ]:
BAD_WEIGHT_LIST = [1.0, 1.5, 2.0, 2.5, 3.0, 3.5, 4.0, 4.5, 5.0]
MODEL_NAMES = ['ResNet-18', 'MobileNet-V2', 'VGG-16']

In [ ]:
resnet_best_weight, resnet_weight_f2, resnet_db =  run_grid_search_with_kfold_cv(
    model_name='resnet18',
    all_paths=all_paths,
    all_labels_np=all_labels_np,
    bad_weight_list=BAD_WEIGHT_LIST,
    n_splits=5,
    num_epochs=40,
    batch_size=16,
    num_workers=NUM_WORKERS,
    device=device,
    run_dirs=run_dirs,
    early_stop_patience=10,
    champion_select='mean'
)

In [ ]:
resnet_y_true_test, resnet_y_pred_test, resnet_model, resnet_history = train_with_best_weight('resnet18', resnet_best_weight)

In [ ]:
plot_confusion_matrix(resnet_y_true_test,
                      resnet_y_pred_test,
                      ['good', 'bad'],
                      run_dirs['figures'],
                      model_name='vgg16')

show_gradcam_grid(resnet_model,
                  val_transform,
                  'test',
                  device,
                  run_dirs['figures'],
                  model_name='vgg16')

In [ ]:
mobilenet_best_weight, mobilenet_weight_f2, mobilenet_db =  run_grid_search_with_kfold_cv(
    model_name='mobilnet_v2',
    all_paths=all_paths,
    all_labels_np=all_labels_np,
    bad_weight_list=BAD_WEIGHT_LIST,
    n_splits=5,
    num_epochs=40,
    batch_size=16,
    num_workers=NUM_WORKERS,
    device=device,
    run_dirs=run_dirs,
    early_stop_patience=10,
    champion_select='mean'
)

In [ ]:
mobilenet_y_true_test, mobilenet_y_pred_test, mobilenet_model, mobilenet_history = train_with_best_weight('resnet18', mobilenet_best_weight)

In [ ]:
plot_confusion_matrix(mobilenet_y_true_test,
                      mobilenet_y_pred_test,
                      ['good', 'bad'],
                      run_dirs['figures'],
                      model_name='mobilenet_v2')

show_gradcam_grid(mobilenet_model,
                  val_transform,
                  'test',
                  device,
                  run_dirs['figures'],
                  model_name='mobilenet_v2')

In [ ]:
vgg_best_weight, vgg_weight_f2, vgg_db =  run_grid_search_with_kfold_cv(
    model_name='vgg16',
    all_paths=all_paths,
    all_labels_np=all_labels_np,
    bad_weight_list=BAD_WEIGHT_LIST,
    n_splits=5,
    num_epochs=40,
    batch_size=16,
    num_workers=NUM_WORKERS,
    device=device,
    run_dirs=run_dirs,
    early_stop_patience=10,
    champion_select='mean'
)

In [ ]:
vgg_y_true_test, vgg_y_pred_test, vgg_model, vgg_history = train_with_best_weight('resnet18', vgg_best_weight)

In [ ]:
plot_confusion_matrix(vgg_y_true_test,
                      vgg_y_pred_test,
                      ['good', 'bad'],
                      run_dirs['figures'],
                      model_name='vgg16')

show_gradcam_grid(vgg_model,
                  val_transform,
                  'test',
                  device,
                  run_dirs['figures'],
                  model_name='vgg16')

In [ ]:
results_vgg = {
    'weight_f2': vgg_weight_f2,
    'db': vgg_db,
    'best_weight': vgg_best_weight
}

results_resnet = {
    'weight_f2': resnet_weight_f2,
    'db': resnet_db,
    'best_weight': resnet_best_weight
}

results_mobilenet = {
    'weight_f2': mobilenet_weight_f2,
    'db': mobilenet_db,
    'best_weight': mobilenet_best_weight
}

all_models_results = {
    'VGG-16': results_vgg,
    'ResNet-18': results_resnet,
    'MobileNet-V2': results_mobilenet
}

plot_all_models_weight_vs_f2(all_models_results, save_path=run_dirs['figures'])

In [ ]:
import random
import numpy as np

def full_train_test_split(all_paths, all_labels_np, seed):
    # 1. 파라미터로 받은 seed 값 적용 (고정된 42 대신 변수 사용)
    random.seed(seed)
    
    CLASS_NAMES = ["good", "type1", "type2", "type3", "type4", "type5"]
    
    # 민엽님이 원하시는 정확한 테스트 셋 개수
    TARGET_TEST_COUNTS = {
        'good': 36, 'type1': 4, 'type2': 4, 
        'type3': 5, 'type4': 3, 'type5': 3
    }
    
    # 2. 클래스별로 파일 경로 분류하기
    paths_by_class = {c: [] for c in CLASS_NAMES}
    for p in all_paths:
        for c in CLASS_NAMES:
            if c in p:
                paths_by_class[c].append(p)
                break
                
    train_paths, test_paths = [], []
    train_labels, test_labels = [], []
    
    # 3. 각 클래스별로 리스트를 섞고, 원하는 개수만큼 정확히 자르기
    for class_name in CLASS_NAMES:
        paths = paths_by_class[class_name].copy()
        random.shuffle(paths) # 시드에 맞춰 무작위로 섞기
        
        n_test = TARGET_TEST_COUNTS[class_name]
        
        test_p = paths[:n_test]
        train_p = paths[n_test:]
        
        test_paths.extend(test_p)
        train_paths.extend(train_p)
        
        # 라벨링 (good은 0, 나머지는 1)
        label = 0 if class_name == 'good' else 1
        test_labels.extend([label] * len(test_p))
        train_labels.extend([label] * len(train_p))
        
    # 기존 sklearn의 train_test_split과 똑같이 4개의 결과값을 반환
    return train_paths, test_paths, np.array(train_labels), np.array(test_labels)

# ── [실제 사용 예시] ────────────────────────────────────────
# 기존 train_test_split 코드 지우고 이렇게 딱 한 줄만 쓰시면 됩니다!
# train_paths, test_paths, train_labels_np, test_labels_np = full_train_test_split(all_paths, all_labels_np, seed=42)

In [ ]:
def custom_Stratified_K_Fold(full_train_paths, n_splits=5, seed=42):
    train_detailed_labels = []
    for p in full_train_paths:
        if 'good' in p: train_detailed_labels.append('good')
        elif 'type1' in p: train_detailed_labels.append('type1')
        elif 'type2' in p: train_detailed_labels.append('type2')
        elif 'type3' in p: train_detailed_labels.append('type3')
        elif 'type4' in p: train_detailed_labels.append('type4')
        elif 'type5' in p: train_detailed_labels.append('type5')

    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    fold_splits = list(skf.split(full_train_paths, train_detailed_labels))
    
    return fold_splits, train_detailed_labels